<div align="center">

# 🌸 CBS5502 —  Computational Linguistics and NLP Technologies
## 🏗️ Semantic Role Labeling (SRL)   
---

### 👨‍🏫 **Instructor**  
**Dr. WAN Mingyu**

### 👨‍🏫 **Teaching Assistant**  
**Mr. BAO Xiaoyi**

✨ *Understanding meaning beyond words.*  
✨ *Teaching machines who did what to whom — and how.*

</div>

---

## 📖 Tutorial Goal

In this session, we explore:

> How can a machine understand **who did what to whom, when, where, and why**?

This is the goal of **Semantic Role Labeling (SRL)**.

Instead of only parsing grammar, SRL uncovers the **semantic structure** behind sentences — enabling deeper reasoning, question answering, summarization, and dialogue systems.

---

## 🌏 Why This Matters

Consider the Mandarin sentence:

> 张三把书给了李四。  
> *(Zhang San gave the book to Li Si.)*

A syntactic parser tells us the structure.  
An SRL system tells us:

| Role | Argument |
|------|----------|
| Agent | 张三 |
| Theme | 书 |
| Recipient | 李四 |
| Predicate | 给 |

This semantic representation is what powers:

- 🤖 Conversational AI  
- 🔎 Information extraction  
- 📚 Machine translation  
- 🧠 Knowledge graph construction  

---

# 🗺️ Tutorial Roadmap

| Phase | Focus | Outcome |
|-------|--------|---------|
| 1️⃣ | Theoretical Foundation | Understand Frame Semantics |
| 2️⃣ | Data & Annotation | Explore Chinese FrameNet |
| 3️⃣ | Rule-Based SRL | Build a baseline system |
---

# 🧠 Part 1 — Theoretical Foundation

## 1.1 What is Frame Semantics?

Frame Semantics (Fillmore, 1976) proposes:

> Words evoke structured background knowledge — called *frames*.

For example, the verb **买 (buy)** evokes a *Commerce_buy* frame:

| Frame Element | Description |
|--------------|-------------|
| Buyer | The purchaser |
| Seller | The provider |
| Goods | What is bought |
| Money | Payment |

SRL operationalizes this theory computationally.

---

# 🛠️ Part 2 — Setting Up the Environment

In [ ]:
# Install spaCy and spacy-stanza for English and Mandarin parsing
!pip install spacy stanza spacy-stanza

# Download the English spaCy model
!python -m spacy download en_core_web_sm

# Download the Mandarin stanza model
#!python -m spacy_stanza.download zh

#!pip install stanza spacy spacy-stanza --upgrade

#!rm -rf ~/.cache/stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.2/881.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
import spacy_stanza

# Load English model
nlp_en = spacy.load("en_core_web_sm")

# Import stanza and download the Mandarin model
#import stanza
#stanza.download("zh")  # Downloads the Chinese model

# Load the Mandarin pipeline into spacy-stanza
#nlp_zh = spacy_stanza.load_pipeline("zh")


In [ ]:
def extract_roles_en(doc):
    """Extract semantic roles for move verbs in English sentences."""
    roles = {"Agent": None, "Theme": None, "Source": None, "Goal": None}

    for token in doc:
        # Identify the Agent (subject of the verb)
        if token.dep_ == "nsubj" and token.head.pos_ == "VERB":
            roles["Agent"] = token.text

        # Identify the Theme (direct object of the verb)
        if token.dep_ == "dobj" and token.head.pos_ == "VERB":
            roles["Theme"] = token.text

        # Identify Source and Goal (prepositional phrases)
        if token.dep_ == "prep" and token.text == "from":
            roles["Source"] = " ".join([child.text for child in token.children if child.dep_ == "pobj"])
        if token.dep_ == "prep" and token.text == "to":
            roles["Goal"] = " ".join([child.text for child in token.children if child.dep_ == "pobj"])

    return roles
'''
def extract_roles_zh(doc):
    """Extract semantic roles for move verbs in Mandarin sentences."""
    roles = {"Agent": None, "Theme": None, "Source": None, "Goal": None}

    for token in doc:
        # Identify the Agent (subject of the verb)
        if token.dep_ == "nsubj" and token.head.pos_ == "VERB":
            roles["Agent"] = token.text

        # Identify the Theme (direct object of the verb)
        if token.dep_ == "obj" and token.head.pos_ == "VERB":
            roles["Theme"] = token.text

        # Identify Source and Goal (prepositional phrases using markers like 从 and 到)
        if token.text == "从":  # Source marker
            roles["Source"] = " ".join([child.text for child in token.children if child.dep_ == "pobj"])
        if token.text == "到":  # Goal marker
            roles["Goal"] = " ".join([child.text for child in token.children if child.dep_ == "pobj"])

    return roles
  '''

'\ndef extract_roles_zh(doc):\n    """Extract semantic roles for move verbs in Mandarin sentences."""\n    roles = {"Agent": None, "Theme": None, "Source": None, "Goal": None}\n\n    for token in doc:\n        # Identify the Agent (subject of the verb)\n        if token.dep_ == "nsubj" and token.head.pos_ == "VERB":\n            roles["Agent"] = token.text\n\n        # Identify the Theme (direct object of the verb)\n        if token.dep_ == "obj" and token.head.pos_ == "VERB":\n            roles["Theme"] = token.text\n\n        # Identify Source and Goal (prepositional phrases using markers like 从 and 到)\n        if token.text == "从":  # Source marker\n            roles["Source"] = " ".join([child.text for child in token.children if child.dep_ == "pobj"])\n        if token.text == "到":  # Goal marker\n            roles["Goal"] = " ".join([child.text for child in token.children if child.dep_ == "pobj"])\n\n    return roles\n  '

In [ ]:
# Test sentence in English
sentence_en = "John moved the box from the table to the floor."

# Parse the sentence
doc_en = nlp_en(sentence_en)

# Extract semantic roles
roles_en = extract_roles_en(doc_en)
print("English Roles:", roles_en)

English Roles: {'Agent': 'John', 'Theme': 'box', 'Source': 'table', 'Goal': 'floor'}


In [ ]:
def identify_construction(doc, verb="move"):
    """Identify constructional patterns associated with move verbs."""
    for token in doc:
        if token.lemma_ == verb:
            # Look for "from ... to ..." patterns
            source = None
            goal = None
            for child in token.children:
                if child.dep_ == "prep" and child.text == "from":
                    source = " ".join([c.text for c in child.children if c.dep_ == "pobj"])
                if child.dep_ == "prep" and child.text == "to":
                    goal = " ".join([c.text for c in child.children if c.dep_ == "pobj"])
            return {"Verb": token.text, "Source": source, "Goal": goal}
    return None

# Combine roles and constructions
def process_sentence_en(sentence):
    doc = nlp_en(sentence)
    roles = extract_roles_en(doc)
    construction = identify_construction(doc)
    return {"Roles": roles, "Construction": construction}

# Test the pipeline
result = process_sentence_en("John moved the box from the table to the floor.")
print(result)

{'Roles': {'Agent': 'John', 'Theme': 'box', 'Source': 'table', 'Goal': 'floor'}, 'Construction': {'Verb': 'moved', 'Source': 'table', 'Goal': 'floor'}}


In [ ]:
from spacy import displacy

# Render the English dependency tree
displacy.render(doc_en, style="dep", jupyter=True)

# Render the Mandarin dependency tree
#displacy.render(doc_zh, style="dep", jupyter=True)

In [ ]:
'''
# Test sentence in Mandarin
sentence_zh = "他把箱子从桌子搬到了地上。"  # Translation: "He moved the box from the table to the ground."

# Parse the Mandarin sentence
doc_zh = nlp_zh(sentence_zh)

# Extract semantic roles
roles_zh = extract_roles_zh(doc_zh)
print("Mandarin Roles:", roles_zh)
'''

'\n# Test sentence in Mandarin\nsentence_zh = "他把箱子从桌子搬到了地上。"  # Translation: "He moved the box from the table to the ground."\n\n# Parse the Mandarin sentence\ndoc_zh = nlp_zh(sentence_zh)\n\n# Extract semantic roles\nroles_zh = extract_roles_zh(doc_zh)\nprint("Mandarin Roles:", roles_zh)\n'

In [ ]:
# Define the semantic role labeler
def extract_roles_chinese(sentence):
    """
    Extract semantic roles for move verbs in a Chinese sentence.
    :param sentence: A Chinese sentence containing a move verb.
    :return: A dictionary with semantic roles: Agent, Theme, Source, Goal.
    """
    roles = {"Agent": None, "Theme": None, "Source": None, "Goal": None}

    # Split sentence into tokens manually (basic tokenization for Chinese)
    tokens = list(sentence)  # Each character is treated as a token

    # Identify the Agent (subject before the verb "把")
    if "把" in tokens:
        ba_index = tokens.index("把")
        roles["Agent"] = "".join(tokens[:ba_index])  # Everything before "把"

    # Identify the Theme (object after "把" but before "从" or a verb)
    if "把" in tokens:
        ba_index = tokens.index("把")
        for i in range(ba_index + 1, len(tokens)):
            if tokens[i] in ["从", "搬", "到"]:  # Stop at prepositions or verbs
                roles["Theme"] = "".join(tokens[ba_index + 1:i])
                break

    # Identify the Source (introduced by "从")
    if "从" in tokens:
        cong_index = tokens.index("从")
        for i in range(cong_index + 1, len(tokens)):
            if tokens[i] in ["搬", "到"]:  # Stop at verbs or prepositions
                roles["Source"] = "".join(tokens[cong_index + 1:i])
                break

    # Identify the Goal (introduced by "到")
    if "到" in tokens:
        dao_index = tokens.index("到")
        for i in range(dao_index + 1, len(tokens)):
            if tokens[i] in ["。"]:  # Stop at the end of the sentence
                roles["Goal"] = "".join(tokens[dao_index + 1:i])
                break

    return roles

# Test the function with a sample sentence
sentence = "他把箱子从桌子搬到了地上。"  # "He moved the box from the table to the ground."
roles = extract_roles_chinese(sentence)
print("Semantic Roles:", roles)

Semantic Roles: {'Agent': '他', 'Theme': '箱子', 'Source': '桌子', 'Goal': '了地上'}


---

<div align="center">

# 🧠 CBS5502 — Practice Exercise  
## 🌟 Basic Semantic Role Labeling (English)

✨ *Understanding meaning beyond syntax.*  
✨ *Who did what to whom — and how?*

</div>

---

# 🎯 Objective

In this exercise, you will manually identify:

- ✅ Predicates  
- ✅ Arguments  
- ✅ Semantic roles (Agent, Patient, Theme, Experiencer, Instrument, Recipient, etc.)

The goal is to strengthen your understanding of **predicate–argument structure** before implementing automated SRL systems.

---

# 📘 Part 1 — Warm-Up (Single Predicate Sentences)

For each sentence:

1. Identify the **main predicate**
2. List all arguments
3. Assign appropriate semantic roles

---

### 📝 Sentence 1

> The chef cooked a delicious meal.

| Role | Argument |
|------|----------|
| Predicate |          |
| Agent |          |
| Theme/Patient |          |

---

### 📝 Sentence 2

> Sarah sent her friend a birthday card.

| Role | Argument |
|------|----------|
| Predicate |          |
| Agent |          |
| Recipient |          |
| Theme |          |

---

### 📝 Sentence 3

> The storm destroyed several houses.

| Role | Argument |
|------|----------|
| Predicate |          |
| Cause |          |
| Patient |          |

---

# 📘 Part 2 — Multiple Arguments

Now identify additional roles such as:

- Instrument
- Location
- Time
- Beneficiary

---

### 📝 Sentence 4

> John cut the cake with a knife.

| Role | Argument |
|------|----------|
| Predicate |          |
| Agent |          |
| Theme |          |
| Instrument |          |

---

### 📝 Sentence 5

> The teacher gave the students homework yesterday.

| Role | Argument |
|------|----------|
| Predicate |          |
| Agent |          |
| Recipient |          |
| Theme |          |
| Time |          |

---

# 📘 Part 3 — Experiencer & Psychological Verbs

These verbs do not always behave like action verbs.

---

### 📝 Sentence 6

> Emily loves classical music.

| Role | Argument |
|------|----------|
| Predicate |          |
| Experiencer |          |
| Stimulus |          |

---

### 📝 Sentence 7

> The movie frightened the children.

| Role | Argument |
|------|----------|
| Predicate |          |
| Stimulus |          |
| Experiencer |          |

---

# 📘 Part 4 — Challenging Cases

Now consider ambiguity and complex structure.

---

### 📝 Sentence 8

> After finishing the project, David emailed the report to his manager.

Questions:

- How many predicates are there?
- Are there implicit arguments?
- Identify roles for each predicate separately.

---

### 📝 Sentence 9

> The book was written by the author in 1995.

- Is this active or passive?
- Who is the Agent?
- What is the Theme?
- What role does “in 1995” play?

---

# 🔍 Reflection Questions

1. Does syntactic subject always equal Agent?
2. How do passive constructions affect role assignment?
3. What roles appear most frequently?
4. How would a computer identify these roles automatically?

---

# 🏆 Bonus Challenge

Rewrite one sentence in passive voice and relabel the roles.

Example:

Active:
> The scientist conducted the experiment.

Passive:
> The experiment was conducted by the scientist.

Do the semantic roles change? Why or why not?

---

<div align="center">

## ✨ Key Insight

Syntactic structure tells us **form**.  
Semantic roles tell us **meaning**.

Mastering this distinction is essential before building neural SRL systems.

</div>

---

## ✅ Practice Exercise Answers

### Sentence 1
- Predicate: **cooked**
- Agent: **the chef**
- Theme/Patient: **a delicious meal**

### Sentence 2
- Predicate: **sent**
- Agent: **Sarah**
- Recipient: **her friend**
- Theme: **a birthday card**

### Sentence 3
- Predicate: **destroyed**
- Cause: **the storm**
- Patient: **several houses**

### Sentence 4
- Predicate: **cut**
- Agent: **John**
- Theme: **the cake**
- Instrument: **a knife**

### Sentence 5
- Predicate: **gave**
- Agent: **the teacher**
- Recipient: **the students**
- Theme: **homework**
- Time: **yesterday**

### Sentence 6
- Predicate: **loves**
- Experiencer: **Emily**
- Stimulus: **classical music**

### Sentence 7
- Predicate: **frightened**
- Stimulus: **the movie**
- Experiencer: **the children**

### Sentence 8
- Predicates: **finishing**, **emailed**
- `finishing`: Agent (implicit) = **David**, Theme = **the project**
- `emailed`: Agent = **David**, Theme = **the report**, Recipient = **his manager**

### Sentence 9
- Passive sentence
- Predicate: **written**
- Agent: **the author**
- Theme: **the book**
- Time: **in 1995**

### Reflection
- Syntactic subject is not always the Agent (especially in passive voice).
- Passive changes surface syntax, but semantic roles remain the same.
- Frequent roles in these examples: Agent, Theme/Patient, Recipient.
- Automatic SRL usually combines syntax + sequence labeling + contextual embeddings.

---

<div align="center">

# 🚀 Extend Your SRL Application in your group project  
### CBS5502 — Advancing Semantic Role Labeling (English Focus)

✨ *From foundational SRL to research-grade English NLP systems.*  
✨ *Transform linguistic theory into scalable AI applications.*

</div>

---

## 🌱 Ways to Extend Your Application (English Resources)

### 🧩 1️⃣ Expand Frame Coverage (English FrameNet)

- Integrate additional frames from **Berkeley FrameNet**
- Model:
  - Motion
  - Commerce
  - Communication
  - Causation
  - Perception
- Incorporate frame hierarchies and inheritance relations
- Handle verb polysemy (e.g., *run* in different frames)

> 💡 Research challenge: How do we automatically disambiguate frame evocation?

---

### 🤖 2️⃣ Upgrade to Neural SRL (English Models)

Move beyond rule-based heuristics:

- Fine-tune **BERT-base-uncased**
- Use **RoBERTa-large**
- Implement:
  - Predicate identification
  - Argument span detection
  - Role classification

Explore architectures such as:
- BiLSTM-CRF
- Transformer-based span models
- Joint predicate-argument models

> 🔬 Compare performance between PropBank-style and FrameNet-style SRL.

---

### 🌐 3️⃣ Build an Interactive English SRL Web App

Deploy your SRL model using:

- ✅ **Streamlit** (fast deployment)
- ✅ **Flask / FastAPI**
- ✅ Visualization tools (spaCy displaCy, custom highlighting)

Features to include:
- Text input
- Predicate highlighting
- Role color-coding
- Confidence scores
- Frame display (if using FrameNet)

> 🎯 Goal: Build a semantic analyzer for real-world English texts.

---

# 🎓 Conclusion

✅ You have built a foundational **Semantic Role Labeling system**.  
✅ You understand predicate–argument structures in English.  
✅ You explored both:
- Rule-based approaches  
- Neural transformer-based approaches  

✅ You handled:
- Ambiguity  
- Complex sentences  
- Multi-predicate constructions  

---

## 🌟 What You Can Now Do

You are now able to:

- Represent **who did what to whom**
- Build structured semantic representations
- Connect linguistic theory to machine learning systems
- Design end-to-end English NLP pipelines

These skills support:

- Question Answering systems  
- Information Extraction  
- Summarization  
- Dialogue systems  
- Knowledge Graph construction  

---

# 🔭 Next Steps (English-Focused)

## 📚 Core Resources

- **Berkeley FrameNet**  
  https://framenet.icsi.berkeley.edu

- **Penn Treebank**
- **English PropBank**
- **OntoNotes 5.0**

---

## 📊 Recommended Datasets

- **PropBank**
- **OntoNotes SRL annotations**
- **Universal Proposition Bank**
- **CoNLL-2005 / CoNLL-2012 SRL datasets**

---

## 🤖 Pre-trained Models

- Hugging Face Transformers:
  - `bert-base-uncased`
  - `roberta-large`
  - `deberta-v3`
- AllenNLP English SRL model
- spaCy transformer pipelines

---

# 🧠 Research Extensions

- Compare FrameNet vs PropBank role schemas
- Experiment with zero-shot SRL
- Build cross-domain SRL systems
- Integrate SRL into LLM-based pipelines
- Perform error analysis on long sentences

---

<div align="center">

# 🌍 Final Reflection

By combining:

📚 English annotated corpora  
🧠 Frame Semantics  
🤖 Transformer-based models  
⚙️ Fine-tuning and evaluation  

You can build a **robust, scalable SRL system for English NLP applications.**

---

### CBS5502 — Bridging Linguistic Theory and AI Practice  
### Your research journey continues.

</div>

---